# 12 — Final Results Synthesis

## Objective

This notebook consolidates the complete radar bird–drone classification study
into a reproducible set of final tables, figures, and scientific conclusions.

It brings together results from:

- real-data learning-curve experiments;
- transformation-based synthetic-data generation and validation;
- real-only versus synthetic-augmented classification;
- five-seed robustness analysis;
- synthetic-to-real ratio ablation;
- session-independent generalization.

## Analysis policy

1. Use only artifacts already produced by earlier notebooks.
2. Do not train or modify any model.
3. Do not select new thresholds.
4. Do not perform new validation or test inference.
5. Preserve the previously locked experimental results.
6. Treat comparisons across different test partitions as descriptive.
7. Save all final tables and figures in a dedicated synthesis directory.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Final synthesis safety locks.
ALLOW_MODEL_TRAINING = False
ALLOW_TEST_INFERENCE = False
ALLOW_THRESHOLD_SELECTION = False

assert ALLOW_MODEL_TRAINING is False
assert ALLOW_TEST_INFERENCE is False
assert ALLOW_THRESHOLD_SELECTION is False

PROJECT_ROOT = Path("..")

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
)

FINAL_SYNTHESIS_DIR = (
    OUTPUT_ROOT
    / "final_results_synthesis"
)

FINAL_TABLE_DIR = (
    FINAL_SYNTHESIS_DIR
    / "tables"
)

FINAL_FIGURE_DIR = (
    FINAL_SYNTHESIS_DIR
    / "figures"
)

FINAL_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

pd.set_option(
    "display.max_columns",
    100
)

pd.set_option(
    "display.width",
    160
)

sns.set_theme(
    style="whitegrid",
    context="notebook"
)

print(
    "FINAL SYNTHESIS SAFE MODE"
)

print(
    "Model training:",
    ALLOW_MODEL_TRAINING
)

print(
    "Test inference:",
    ALLOW_TEST_INFERENCE
)

print(
    "Threshold selection:",
    ALLOW_THRESHOLD_SELECTION
)

print(
    "Final synthesis directory:",
    FINAL_SYNTHESIS_DIR.resolve()
)
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Final synthesis safety locks.
ALLOW_MODEL_TRAINING = False
ALLOW_TEST_INFERENCE = False
ALLOW_THRESHOLD_SELECTION = False

assert ALLOW_MODEL_TRAINING is False
assert ALLOW_TEST_INFERENCE is False
assert ALLOW_THRESHOLD_SELECTION is False

PROJECT_ROOT = Path("..")

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
)

FINAL_SYNTHESIS_DIR = (
    OUTPUT_ROOT
    / "final_results_synthesis"
)

FINAL_TABLE_DIR = (
    FINAL_SYNTHESIS_DIR
    / "tables"
)

FINAL_FIGURE_DIR = (
    FINAL_SYNTHESIS_DIR
    / "figures"
)

FINAL_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

pd.set_option(
    "display.max_columns",
    100
)

pd.set_option(
    "display.width",
    160
)

sns.set_theme(
    style="whitegrid",
    context="notebook"
)

print(
    "FINAL SYNTHESIS SAFE MODE"
)

print(
    "Model training:",
    ALLOW_MODEL_TRAINING
)

print(
    "Test inference:",
    ALLOW_TEST_INFERENCE
)

print(
    "Threshold selection:",
    ALLOW_THRESHOLD_SELECTION
)

print(
    "Final synthesis directory:",
    FINAL_SYNTHESIS_DIR.resolve()
)

## 1. Artifact Discovery and Integrity

The synthesis begins in read-only mode by locating the saved outputs from the preceding experiments. Initial candidate paths are inspected, corrected where necessary, and reduced to a locked set of summary artifacts. Per-run predictions, model checkpoints, and training tensors are not loaded.

In [ ]:
EXPERIMENT_DIRECTORIES = {
    "baseline_classification":
        OUTPUT_ROOT
        / "baseline_classification",

    "real_data_learning_curve":
        OUTPUT_ROOT
        / "real_data_learning_curve",

    "synthetic_data_generation":
        OUTPUT_ROOT
        / "synthetic_data_generation",

    "synthetic_augmentation":
        OUTPUT_ROOT
        / "synthetic_augmentation_classification",

    "multiseed_robustness":
        OUTPUT_ROOT
        / "multiseed_augmentation_robustness",

    "synthetic_ratio_ablation":
        OUTPUT_ROOT
        / "synthetic_ratio_ablation",

    "session_independent":
        OUTPUT_ROOT
        / "session_independent_generalization"
}

artifact_inventory_records = []

for (
    experiment_name,
    experiment_directory
) in EXPERIMENT_DIRECTORIES.items():
    directory_exists = (
        experiment_directory.exists()
    )

    csv_files = (
        sorted(
            experiment_directory.rglob(
                "*.csv"
            )
        )
        if directory_exists
        else []
    )

    json_files = (
        sorted(
            experiment_directory.rglob(
                "*.json"
            )
        )
        if directory_exists
        else []
    )

    png_files = (
        sorted(
            experiment_directory.rglob(
                "*.png"
            )
        )
        if directory_exists
        else []
    )

    artifact_inventory_records.append({
        "experiment":
            experiment_name,

        "directory":
            str(
                experiment_directory
            ),

        "directory_exists":
            directory_exists,

        "csv_files":
            len(
                csv_files
            ),

        "json_files":
            len(
                json_files
            ),

        "png_files":
            len(
                png_files
            ),

        "total_result_files":
            (
                len(csv_files)
                + len(json_files)
                + len(png_files)
            )
    })

artifact_inventory_df = pd.DataFrame(
    artifact_inventory_records
)

display(
    artifact_inventory_df
)

missing_experiment_directories = (
    artifact_inventory_df.loc[
        ~artifact_inventory_df[
            "directory_exists"
        ],
        "experiment"
    ].tolist()
)

if missing_experiment_directories:
    print(
        "Directories not found:",
        missing_experiment_directories
    )

    print(
        "This does not yet indicate an error; "
        "the directory may use a different name."
    )

else:
    print(
        "All expected experiment directories "
        "were found."
    )

artifact_inventory_df.to_csv(
    FINAL_TABLE_DIR
    / "artifact_directory_inventory.csv",
    index=False
)

In [ ]:
# Read-only discovery of the actual
# top-level output directory names.

output_directory_records = []

for path in sorted(
    OUTPUT_ROOT.iterdir()
):
    if not path.is_dir():
        continue

    csv_count = len(
        list(
            path.rglob("*.csv")
        )
    )

    json_count = len(
        list(
            path.rglob("*.json")
        )
    )

    png_count = len(
        list(
            path.rglob("*.png")
        )
    )

    output_directory_records.append({
        "directory_name":
            path.name,

        "relative_path":
            str(path),

        "csv_files":
            csv_count,

        "json_files":
            json_count,

        "png_files":
            png_count,

        "total_result_files":
            (
                csv_count
                + json_count
                + png_count
            )
    })

available_output_directories_df = (
    pd.DataFrame(
        output_directory_records
    )
)

display(
    available_output_directories_df
    .sort_values(
        "directory_name"
    )
    .reset_index(drop=True)
)

In [ ]:
DISCOVERY_KEYWORDS = [
    "learning",
    "curve",
    "sample",
    "fraction",
    "synthetic",
    "generation",
    "quality",
    "manifest"
]

candidate_artifact_records = []

for path in OUTPUT_ROOT.rglob("*"):
    if (
        not path.is_file()
        or path.suffix.lower()
        not in {
            ".csv",
            ".json",
            ".png"
        }
    ):
        continue

    normalized_path = str(
        path
    ).lower()

    matched_keywords = [
        keyword
        for keyword in DISCOVERY_KEYWORDS
        if keyword in normalized_path
    ]

    if matched_keywords:
        candidate_artifact_records.append({
            "file_name":
                path.name,

            "relative_path":
                str(path),

            "file_type":
                path.suffix.lower(),

            "matched_keywords":
                ", ".join(
                    matched_keywords
                )
        })

candidate_artifacts_df = (
    pd.DataFrame(
        candidate_artifact_records
    )
)

if candidate_artifacts_df.empty:
    print(
        "No matching output artifacts "
        "were found."
    )

else:
    display(
        candidate_artifacts_df
        .sort_values([
            "relative_path",
            "file_name"
        ])
        .reset_index(drop=True)
    )

print(
    "Candidate artifacts found:",
    len(
        candidate_artifacts_df
    )
)

In [ ]:
SYNTHETIC_DATA_ROOT = (
    PROJECT_ROOT
    / "data"
    / "synthetic"
)

synthetic_storage_records = []

if SYNTHETIC_DATA_ROOT.exists():
    for path in sorted(
        SYNTHETIC_DATA_ROOT.rglob("*")
    ):
        if (
            path.is_file()
            and path.suffix.lower()
            in {
                ".csv",
                ".json",
                ".npy",
                ".png"
            }
        ):
            synthetic_storage_records.append({
                "file_name":
                    path.name,

                "relative_path":
                    str(path),

                "file_type":
                    path.suffix.lower(),

                "size_mb":
                    (
                        path.stat().st_size
                        / (
                            1024 ** 2
                        )
                    )
            })

synthetic_storage_df = pd.DataFrame(
    synthetic_storage_records
)

if synthetic_storage_df.empty:
    print(
        "No synthetic-data artifacts were "
        "found under:",
        SYNTHETIC_DATA_ROOT
    )

else:
    display(
        synthetic_storage_df
        .sort_values(
            "relative_path"
        )
        .reset_index(drop=True)
        .style.format({
            "size_mb": "{:.3f}"
        })
    )

print(
    "Synthetic storage files found:",
    len(
        synthetic_storage_df
    )
)

In [ ]:
EXPERIMENT_DIRECTORIES = {
    "baseline_classification":
        OUTPUT_ROOT
        / "baseline_classification",

    # Notebook 06 saved its final tables
    # inside baseline_classification.
    "real_data_learning_curve":
        OUTPUT_ROOT
        / "baseline_classification",

    # Notebook 07 uses this directory name.
    "synthetic_data_validation":
        OUTPUT_ROOT
        / "synthetic_data_validation",

    "synthetic_augmentation":
        OUTPUT_ROOT
        / "synthetic_augmentation_classification",

    "multiseed_robustness":
        OUTPUT_ROOT
        / "multiseed_augmentation_robustness",

    "synthetic_ratio_ablation":
        OUTPUT_ROOT
        / "synthetic_ratio_ablation",

    "session_independent":
        OUTPUT_ROOT
        / "session_independent_generalization"
}

for (
    experiment_name,
    experiment_directory
) in EXPERIMENT_DIRECTORIES.items():
    if not experiment_directory.exists():
        raise FileNotFoundError(
            f"{experiment_name}: directory "
            f"not found: "
            f"{experiment_directory.resolve()}"
        )

print(
    "All corrected experiment "
    "directories were found."
)

In [ ]:
def resolve_unique_artifact(
    experiment_directory,
    file_name
):
    matches = sorted(
        experiment_directory.rglob(
            file_name
        )
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Artifact not found: {file_name}\n"
            f"Search directory: "
            f"{experiment_directory.resolve()}"
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Expected one artifact named "
            f"{file_name}, found "
            f"{len(matches)}:\n"
            + "\n".join(
                str(path.resolve())
                for path in matches
            )
        )

    return matches[0]


FINAL_ARTIFACT_PATHS = {
    # Notebook 06
    "learning_curve":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "real_data_learning_curve"
            ],
            "real_data_learning_curve.csv"
        ),

    "learning_curve_incremental":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "real_data_learning_curve"
            ],
            "real_data_incremental_improvements.csv"
        ),

    # Notebook 07
    "synthetic_quality":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_data_validation"
            ],
            "quality_summary.csv"
        ),

    "synthetic_similarity":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_data_validation"
            ],
            "parent_similarity_summary.csv"
        ),

    "synthetic_alignment":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_data_validation"
            ],
            "alignment_summary.csv"
        ),

    "synthetic_distribution":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_data_validation"
            ],
            "distribution_summary.csv"
        ),

    # Notebook 08
    "augmentation_baseline_comparison":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_augmentation"
            ],
            "baseline_comparison.csv"
        ),

    "augmentation_improvement":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_augmentation"
            ],
            "augmentation_improvement.csv"
        ),

    "augmentation_subtype_metrics":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_augmentation"
            ],
            "subtype_metrics.csv"
        ),

    "augmentation_range_metrics":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_augmentation"
            ],
            "range_metrics.csv"
        ),

    # Notebook 09
    "multiseed_test_metrics":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "multiseed_robustness"
            ],
            "all_seed_test_metrics.csv"
        ),

    "multiseed_configuration_summary":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "multiseed_robustness"
            ],
            "configuration_summary.csv"
        ),

    "multiseed_paired_improvement":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "multiseed_robustness"
            ],
            "paired_improvement_summary.csv"
        ),

    "multiseed_stability":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "multiseed_robustness"
            ],
            "stability_summary.csv"
        ),

    # Notebook 10
    "ratio_validation_results":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_ratio_ablation"
            ],
            "validation_ratio_results.csv"
        ),

    "ratio_validation_summary":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_ratio_ablation"
            ],
            "validation_ratio_summary.csv"
        ),

    "ratio_efficiency":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_ratio_ablation"
            ],
            "ratio_efficiency_summary.csv"
        ),

    "ratio_selection_manifest":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_ratio_ablation"
            ],
            "ratio_selection_manifest.json"
        ),

    # Notebook 11
    "session_test_metrics":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "session_independent"
            ],
            "all_seed_test_metrics.csv"
        ),

    "session_configuration_summary":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "session_independent"
            ],
            "configuration_summary.csv"
        ),

    "session_paired_improvement":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "session_independent"
            ],
            "paired_improvement_summary.csv"
        ),

    "split_protocol_comparison":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "session_independent"
            ],
            "official_vs_session_independent_comparison.csv"
        )
}

final_artifact_inventory_df = pd.DataFrame([
    {
        "artifact":
            artifact_name,

        "file_name":
            artifact_path.name,

        "relative_path":
            str(
                artifact_path
            ),

        "size_kb":
            (
                artifact_path.stat().st_size
                / 1024
            )
    }
    for artifact_name, artifact_path
    in FINAL_ARTIFACT_PATHS.items()
])

display(
    final_artifact_inventory_df
    .style
    .format({
        "size_kb": "{:.2f}"
    })
)

print(
    "Final summary artifacts resolved:",
    len(
        FINAL_ARTIFACT_PATHS
    )
)

In [ ]:
final_result_tables = {}

for (
    artifact_name,
    artifact_path
) in FINAL_ARTIFACT_PATHS.items():
    if artifact_path.suffix.lower() != ".csv":
        continue

    result_table = pd.read_csv(
        artifact_path
    )

    if result_table.empty:
        raise ValueError(
            f"{artifact_name} is empty: "
            f"{artifact_path.resolve()}"
        )

    final_result_tables[
        artifact_name
    ] = result_table

table_structure_records = []

for (
    table_name,
    result_table
) in final_result_tables.items():
    table_structure_records.append({
        "table":
            table_name,

        "rows":
            len(
                result_table
            ),

        "columns":
            len(
                result_table.columns
            ),

        "duplicate_rows":
            int(
                result_table
                .duplicated()
                .sum()
            ),

        "missing_values":
            int(
                result_table
                .isna()
                .sum()
                .sum()
            ),

        "column_names":
            ", ".join(
                result_table.columns
            )
    })

table_structure_df = pd.DataFrame(
    table_structure_records
)

display(
    table_structure_df
)

with open(
    FINAL_ARTIFACT_PATHS[
        "ratio_selection_manifest"
    ],
    "r",
    encoding="utf-8"
) as file:
    ratio_selection_manifest = (
        json.load(file)
    )

print(
    "Loaded CSV result tables:",
    len(
        final_result_tables
    )
)

print(
    "Ratio-selection manifest loaded."
)

print(
    "No model files, predictions, or "
    "training datasets were loaded."
)

In [ ]:
CORE_COMPLETE_TABLES = [
    "synthetic_quality",
    "synthetic_similarity",
    "synthetic_alignment",
    "synthetic_distribution",
    "augmentation_baseline_comparison",
    "augmentation_improvement",
    "multiseed_test_metrics",
    "multiseed_configuration_summary",
    "multiseed_paired_improvement",
    "multiseed_stability",
    "ratio_validation_results",
    "ratio_validation_summary",
    "ratio_efficiency",
    "session_test_metrics",
    "session_configuration_summary",
    "session_paired_improvement",
    "split_protocol_comparison"
]

for table_name in CORE_COMPLETE_TABLES:
    result_table = final_result_tables[
        table_name
    ]

    missing_value_count = int(
        result_table
        .isna()
        .sum()
        .sum()
    )

    if missing_value_count != 0:
        raise ValueError(
            f"{table_name} contains "
            f"{missing_value_count} "
            "unexpected missing values."
        )

assert len(
    final_result_tables[
        "learning_curve"
    ]
) == 4

assert len(
    final_result_tables[
        "multiseed_test_metrics"
    ]
) == 10

assert len(
    final_result_tables[
        "ratio_validation_results"
    ]
) == 15

assert len(
    final_result_tables[
        "session_test_metrics"
    ]
) == 10

assert set(
    final_result_tables[
        "multiseed_test_metrics"
    ]["seed"]
) == {
    42,
    52,
    62,
    72,
    82
}

assert set(
    final_result_tables[
        "session_test_metrics"
    ]["seed"]
) == {
    42,
    52,
    62,
    72,
    82
}

assert set(
    final_result_tables[
        "ratio_validation_results"
    ]["synthetic_ratio"]
) == {
    0.0,
    0.5,
    1.0
}

print(
    "All core result tables passed "
    "the final integrity checks."
)

print(
    "Learning-curve missing values are "
    "retained because boundary comparisons "
    "are not mathematically defined."
)

## 2. Headline Multi-Seed Results

The primary comparison reports five-seed mean and standard deviation for real-only and 1:1 synthetic-augmented training under both the official segment-level protocol and the stricter session-independent protocol. Comparisons between protocols are descriptive because their test observations differ.

In [ ]:
HEADLINE_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "bird_f1",
    "drone_recall",
    "roc_auc"
]

headline_result_records = []

summary_sources = [
    (
        "Official segment-level split",
        final_result_tables[
            "multiseed_configuration_summary"
        ]
    ),
    (
        "Session-independent split",
        final_result_tables[
            "session_configuration_summary"
        ]
    )
]

for (
    evaluation_protocol,
    configuration_summary
) in summary_sources:
    for _, summary_row in (
        configuration_summary.iterrows()
    ):
        result_record = {
            "evaluation_protocol":
                evaluation_protocol,

            "configuration":
                summary_row[
                    "configuration"
                ],

            "configuration_display_name":
                summary_row[
                    "configuration_display_name"
                ],

            "model_seeds":
                5
        }

        for metric_name in HEADLINE_METRICS:
            result_record[
                metric_name + "_mean"
            ] = float(
                summary_row[
                    metric_name + "_mean"
                ]
            )

            result_record[
                metric_name + "_std"
            ] = float(
                summary_row[
                    metric_name + "_std"
                ]
            )

        headline_result_records.append(
            result_record
        )

headline_results_df = pd.DataFrame(
    headline_result_records
)

headline_results_df.to_csv(
    FINAL_TABLE_DIR
    / "headline_multiseed_results.csv",
    index=False
)

display_columns = [
    "evaluation_protocol",
    "configuration_display_name",
    "model_seeds",
    "accuracy_mean",
    "accuracy_std",
    "balanced_accuracy_mean",
    "balanced_accuracy_std",
    "macro_f1_mean",
    "macro_f1_std",
    "bird_f1_mean",
    "bird_f1_std",
    "drone_recall_mean",
    "drone_recall_std",
    "roc_auc_mean",
    "roc_auc_std"
]

display(
    headline_results_df[
        display_columns
    ].style.format({
        column: "{:.4f}"
        for column in display_columns
        if column not in {
            "evaluation_protocol",
            "configuration_display_name",
            "model_seeds"
        }
    })
)

print(
    "Headline multi-seed results saved."
)

In [ ]:
split_protocol_comparison_df = (
    final_result_tables[
        "split_protocol_comparison"
    ].copy()
)

split_protocol_comparison_df[
    "gain_preservation_percent"
] = (
    split_protocol_comparison_df[
        "session_independent_augmentation_gain"
    ]
    / split_protocol_comparison_df[
        "official_augmentation_gain"
    ]
    * 100.0
)

split_protocol_comparison_df[
    "absolute_gain_difference"
] = (
    split_protocol_comparison_df[
        "session_independent_augmentation_gain"
    ]
    - split_protocol_comparison_df[
        "official_augmentation_gain"
    ]
)

gain_preservation_df = (
    split_protocol_comparison_df[
        [
            "metric",
            "metric_display_name",
            "official_augmentation_gain",
            "session_independent_augmentation_gain",
            "absolute_gain_difference",
            "gain_preservation_percent"
        ]
    ]
    .copy()
)

gain_preservation_df.to_csv(
    FINAL_TABLE_DIR
    / "augmentation_gain_preservation.csv",
    index=False
)

display(
    gain_preservation_df.style.format({
        "official_augmentation_gain":
            "{:+.4f}",

        "session_independent_augmentation_gain":
            "{:+.4f}",

        "absolute_gain_difference":
            "{:+.4f}",

        "gain_preservation_percent":
            "{:.2f}%"
    })
)

print(
    "Important: gain preservation is a "
    "descriptive comparison because the "
    "protocols use different test partitions."
)

## 3. Real-Data Learning Curve and Data Efficiency

The real-only learning curve quantifies the effect of increasing the balanced training subset from 10% to 100%. The reference augmented model is then compared with the 10% and 25% real-only baselines to measure how much of the additional-real-data performance gap is recovered by synthetic augmentation.

In [ ]:
learning_curve_df = (
    final_result_tables[
        "learning_curve"
    ]
    .copy()
)

REAL_FRACTION_PERCENT = {
    "10_percent": 10,
    "25_percent": 25,
    "50_percent": 50,
    "100_percent": 100
}

learning_curve_df[
    "real_fraction_percent"
] = (
    learning_curve_df[
        "subset"
    ].map(
        REAL_FRACTION_PERCENT
    )
)

assert (
    learning_curve_df[
        "real_fraction_percent"
    ].notna().all()
)

learning_curve_df = (
    learning_curve_df
    .sort_values(
        "real_fraction_percent"
    )
    .reset_index(drop=True)
)

learning_curve_display_columns = [
    "subset",
    "real_fraction_percent",
    "samples_per_class",
    "training_samples",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "bird_precision",
    "bird_recall",
    "bird_f1",
    "drone_recall",
    "roc_auc"
]

display(
    learning_curve_df[
        learning_curve_display_columns
    ].style.format({
        "real_fraction_percent":
            "{:.0f}%",

        "accuracy":
            "{:.4f}",

        "balanced_accuracy":
            "{:.4f}",

        "macro_f1":
            "{:.4f}",

        "bird_precision":
            "{:.4f}",

        "bird_recall":
            "{:.4f}",

        "bird_f1":
            "{:.4f}",

        "drone_recall":
            "{:.4f}",

        "roc_auc":
            "{:.4f}"
    })
)

learning_curve_df.to_csv(
    FINAL_TABLE_DIR
    / "final_real_data_learning_curve.csv",
    index=False
)

In [ ]:
augmentation_comparison_df = (
    final_result_tables[
        "augmentation_baseline_comparison"
    ]
    .copy()
)

synthetic_model_mask = (
    augmentation_comparison_df[
        "model"
    ]
    .astype(str)
    .str.contains(
        "synthetic",
        case=False,
        regex=False
    )
)

assert (
    synthetic_model_mask.sum()
    == 1
)

augmented_seed_42_row = (
    augmentation_comparison_df.loc[
        synthetic_model_mask
    ]
    .iloc[0]
)

DATA_EFFICIENCY_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "bird_precision",
    "bird_recall",
    "bird_f1",
    "drone_recall",
    "roc_auc"
]

learning_curve_indexed = (
    learning_curve_df
    .set_index(
        "subset"
    )
)

data_efficiency_records = []

for metric_name in DATA_EFFICIENCY_METRICS:
    real_10_score = float(
        learning_curve_indexed.loc[
            "10_percent",
            metric_name
        ]
    )

    real_25_score = float(
        learning_curve_indexed.loc[
            "25_percent",
            metric_name
        ]
    )

    augmented_score = float(
        augmented_seed_42_row[
            metric_name
        ]
    )

    real_10_to_25_gap = (
        real_25_score
        - real_10_score
    )

    recovered_gap_percent = (
        (
            augmented_score
            - real_10_score
        )
        / real_10_to_25_gap
        * 100.0
        if not np.isclose(
            real_10_to_25_gap,
            0.0
        )
        else np.nan
    )

    data_efficiency_records.append({
        "metric":
            metric_name,

        "real_only_10_percent":
            real_10_score,

        "augmented_10_percent_real":
            augmented_score,

        "real_only_25_percent":
            real_25_score,

        "augmentation_gain_over_10_percent":
            augmented_score
            - real_10_score,

        "difference_from_25_percent":
            augmented_score
            - real_25_score,

        "recovered_10_to_25_gap_percent":
            recovered_gap_percent
    })

data_efficiency_df = pd.DataFrame(
    data_efficiency_records
)

data_efficiency_df.to_csv(
    FINAL_TABLE_DIR
    / "synthetic_augmentation_data_efficiency.csv",
    index=False
)

display(
    data_efficiency_df.style.format({
        "real_only_10_percent":
            "{:.4f}",

        "augmented_10_percent_real":
            "{:.4f}",

        "real_only_25_percent":
            "{:.4f}",

        "augmentation_gain_over_10_percent":
            "{:+.4f}",

        "difference_from_25_percent":
            "{:+.4f}",

        "recovered_10_to_25_gap_percent":
            "{:.2f}%"
    })
)

In [ ]:
LEARNING_CURVE_PLOT_METRICS = [
    (
        "balanced_accuracy",
        "Balanced accuracy"
    ),
    (
        "macro_f1",
        "Macro-F1"
    ),
    (
        "bird_f1",
        "Bird F1"
    )
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4.8),
    sharex=True,
    constrained_layout=True
)

for axis, (
    metric_name,
    metric_display_name
) in zip(
    axes,
    LEARNING_CURVE_PLOT_METRICS
):
    axis.plot(
        learning_curve_df[
            "real_fraction_percent"
        ],
        learning_curve_df[
            metric_name
        ],
        marker="o",
        markersize=7,
        linewidth=2.2,
        color="#3569A8",
        label="Real-only"
    )

    axis.scatter(
        [10],
        [
            float(
                augmented_seed_42_row[
                    metric_name
                ]
            )
        ],
        marker="*",
        s=220,
        color="#E67E22",
        edgecolor="black",
        linewidth=0.7,
        zorder=5,
        label="10% real + synthetic 1:1"
    )

    axis.set_title(
        metric_display_name
    )

    axis.set_xlabel(
        "Real training-data fraction"
    )

    axis.set_xticks([
        10,
        25,
        50,
        100
    ])

    axis.set_xticklabels([
        "10%",
        "25%",
        "50%",
        "100%"
    ])

    axis.set_ylim(
        0.60,
        1.01
    )

    axis.grid(
        alpha=0.25
    )

axes[0].set_ylabel(
    "Official test score"
)

axes[0].legend(
    loc="lower right",
    fontsize=9
)

fig.suptitle(
    "Real-Data Learning Curve and "
    "Synthetic-Augmentation Data Efficiency",
    fontsize=14
)

learning_curve_figure_path = (
    FINAL_FIGURE_DIR
    / "learning_curve_and_augmentation.png"
)

fig.savefig(
    learning_curve_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Figure saved:",
    learning_curve_figure_path.resolve()
)

## 4. Synthetic-Data Quality and Distribution Preservation

Quality indicators assess whether the transformations preserve the parent signal structure and class-level intensity, variability, and energy distributions. High parent similarity demonstrates transformation fidelity, not independence from the original physical observations.

In [ ]:
synthetic_quality_df = (
    final_result_tables[
        "synthetic_quality"
    ]
    .copy()
)

synthetic_alignment_df = (
    final_result_tables[
        "synthetic_alignment"
    ]
    .copy()
)

quality_value_map = dict(
    zip(
        synthetic_quality_df[
            "quality_indicator"
        ],
        pd.to_numeric(
            synthetic_quality_df[
                "value"
            ],
            errors="raise"
        )
    )
)

bird_mean_aligned_correlation = float(
    quality_value_map[
        "Bird mean aligned correlation"
    ]
)

drone_mean_aligned_correlation = float(
    quality_value_map[
        "Drone mean aligned correlation"
    ]
)

low_correlation_sample_count = float(
    quality_value_map[
        "Samples below 0.80 aligned correlation"
    ]
)

# Verify high average structural fidelity.
assert (
    bird_mean_aligned_correlation
    > 0.95
)

assert (
    drone_mean_aligned_correlation
    > 0.95
)

# Verify that the low-correlation count is
# a valid non-negative integer-like value.
assert np.isfinite(
    low_correlation_sample_count
)

assert (
    low_correlation_sample_count
    >= 0
)

assert np.isclose(
    low_correlation_sample_count,
    round(
        low_correlation_sample_count
    ),
    rtol=0.0,
    atol=1e-8
)

# The final stored artifact contains no more
# than one sample below the quality threshold.
assert (
    low_correlation_sample_count
    <= 1
)

display(
    synthetic_quality_df.style.format({
        "value": "{:.4f}"
    })
)

display(
    synthetic_alignment_df.style.format({
        column: "{:.4f}"
        for column
        in synthetic_alignment_df.columns
        if column not in {
            "parent_target_group",
            "samples"
        }
    })
)

synthetic_quality_df.to_csv(
    FINAL_TABLE_DIR
    / "final_synthetic_quality_summary.csv",
    index=False
)

synthetic_alignment_df.to_csv(
    FINAL_TABLE_DIR
    / "final_synthetic_alignment_summary.csv",
    index=False
)

print(
    "Synthetic-data quality checks passed."
)

print(
    "Bird mean aligned correlation:",
    round(
        bird_mean_aligned_correlation,
        4
    )
)

print(
    "Drone mean aligned correlation:",
    round(
        drone_mean_aligned_correlation,
        4
    )
)

print(
    "Samples below 0.80 aligned "
    "correlation:",
    int(
        round(
            low_correlation_sample_count
        )
    )
)

print(
    "Interpretation: high aligned similarity "
    "confirms transformation fidelity, but "
    "does not imply that synthetic children "
    "are independent physical observations."
)

In [ ]:
synthetic_distribution_df = (
    final_result_tables[
        "synthetic_distribution"
    ]
    .copy()
)

real_distribution_df = (
    synthetic_distribution_df[
        synthetic_distribution_df[
            "source"
        ] == "real"
    ]
    .copy()
)

generated_distribution_df = (
    synthetic_distribution_df[
        synthetic_distribution_df[
            "source"
        ] == "synthetic"
    ]
    .copy()
)

distribution_comparison_df = (
    real_distribution_df
    .merge(
        generated_distribution_df,
        on="target_group",
        suffixes=(
            "_real",
            "_synthetic"
        ),
        validate="one_to_one"
    )
)

DISTRIBUTION_METRICS = [
    "mean_intensity",
    "mean_standard_deviation",
    "mean_energy"
]

distribution_change_records = []

for _, row in (
    distribution_comparison_df
    .iterrows()
):
    result_record = {
        "target_group":
            row[
                "target_group"
            ]
    }

    for metric_name in DISTRIBUTION_METRICS:
        real_value = float(
            row[
                metric_name
                + "_real"
            ]
        )

        synthetic_value = float(
            row[
                metric_name
                + "_synthetic"
            ]
        )

        result_record[
            metric_name + "_real"
        ] = real_value

        result_record[
            metric_name + "_synthetic"
        ] = synthetic_value

        result_record[
            metric_name
            + "_relative_change_percent"
        ] = (
            (
                synthetic_value
                - real_value
            )
            / real_value
            * 100.0
        )

    distribution_change_records.append(
        result_record
    )

distribution_change_df = pd.DataFrame(
    distribution_change_records
)

distribution_change_df.to_csv(
    FINAL_TABLE_DIR
    / "real_vs_synthetic_distribution.csv",
    index=False
)

display(
    distribution_change_df.style.format({
        column: (
            "{:+.2f}%"
            if column.endswith(
                "_relative_change_percent"
            )
            else "{:.4f}"
        )
        for column
        in distribution_change_df.columns
        if column != "target_group"
    })
)

## 5. Synthetic-to-Real Ratio Ablation

Candidate ratios are compared using validation results only. The locked ratio maximizes mean validation macro-F1, with balanced accuracy and accuracy used as tie-breakers. No additional test inference is performed during synthesis.

In [ ]:
ratio_summary_df = (
    final_result_tables[
        "ratio_validation_summary"
    ]
    .copy()
    .sort_values(
        "synthetic_ratio"
    )
    .reset_index(drop=True)
)

selected_ratio = float(
    ratio_selection_manifest[
        "selected_synthetic_ratio"
    ]
)

assert np.isclose(
    selected_ratio,
    1.0
)

ratio_display_columns = [
    "synthetic_ratio",
    "configuration_display_name",
    "validation_accuracy_mean",
    "validation_accuracy_std",
    "validation_balanced_accuracy_mean",
    "validation_balanced_accuracy_std",
    "validation_macro_f1_mean",
    "validation_macro_f1_std",
    "validation_bird_f1_mean",
    "validation_bird_f1_std",
    "validation_roc_auc_mean",
    "validation_roc_auc_std"
]

display(
    ratio_summary_df[
        ratio_display_columns
    ].style.format({
        "synthetic_ratio":
            "{:.1f}",

        **{
            column: "{:.4f}"
            for column
            in ratio_display_columns
            if column not in {
                "synthetic_ratio",
                "configuration_display_name"
            }
        }
    })
)

ratio_summary_df.to_csv(
    FINAL_TABLE_DIR
    / "final_ratio_ablation_summary.csv",
    index=False
)

RATIO_PLOT_METRICS = [
    (
        "validation_balanced_accuracy",
        "Balanced accuracy"
    ),
    (
        "validation_macro_f1",
        "Macro-F1"
    ),
    (
        "validation_bird_f1",
        "Bird F1"
    ),
    (
        "validation_roc_auc",
        "ROC-AUC"
    )
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11.5, 8.5),
    constrained_layout=True
)

for axis, (
    metric_name,
    metric_display_name
) in zip(
    axes.flat,
    RATIO_PLOT_METRICS
):
    axis.errorbar(
        ratio_summary_df[
            "synthetic_ratio"
        ],
        ratio_summary_df[
            metric_name
            + "_mean"
        ],
        yerr=ratio_summary_df[
            metric_name
            + "_std"
        ],
        marker="o",
        markersize=7,
        linewidth=2,
        capsize=5,
        color="#3569A8"
    )

    axis.axvline(
        selected_ratio,
        color="#E67E22",
        linestyle="--",
        linewidth=1.5,
        label="Selected ratio: 1.0"
    )

    axis.set_title(
        metric_display_name
    )

    axis.set_xlabel(
        "Synthetic-to-real ratio"
    )

    axis.set_ylabel(
        "Mean validation score ± SD"
    )

    axis.set_xticks([
        0.0,
        0.5,
        1.0
    ])

    axis.set_ylim(
        0.50,
        1.01
    )

    axis.grid(
        alpha=0.25
    )

axes[0, 0].legend(
    loc="lower right"
)

fig.suptitle(
    "Validation Performance by "
    "Synthetic-to-Real Ratio",
    fontsize=14
)

ratio_figure_path = (
    FINAL_FIGURE_DIR
    / "synthetic_ratio_ablation.png"
)

fig.savefig(
    ratio_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Locked ratio:",
    selected_ratio
)

print(
    "Ratio selected using validation "
    "results only."
)

print(
    "No new test inference was performed."
)

## 6. Paired Multi-Seed Robustness

Paired seed-level improvements, descriptive confidence intervals, and performance variability summarize robustness to model initialization. With only five seeds, the intervals are descriptive and are not interpreted as definitive significance tests.

In [ ]:
official_paired_df = (
    final_result_tables[
        "multiseed_paired_improvement"
    ]
    .copy()
)

session_paired_df = (
    final_result_tables[
        "session_paired_improvement"
    ]
    .copy()
)

METRIC_DISPLAY_NAMES = {
    "accuracy": "Accuracy",
    "balanced_accuracy": "Balanced accuracy",
    "macro_f1": "Macro-F1",
    "bird_precision": "Bird precision",
    "bird_recall": "Bird recall",
    "bird_f1": "Bird F1",
    "drone_recall": "Drone recall",
    "roc_auc": "ROC-AUC"
}

if (
    "metric_display_name"
    not in official_paired_df.columns
):
    official_paired_df[
        "metric_display_name"
    ] = (
        official_paired_df[
            "metric"
        ].map(
            METRIC_DISPLAY_NAMES
        )
    )

official_paired_df[
    "evaluation_protocol"
] = "Official segment-level split"

session_paired_df[
    "evaluation_protocol"
] = "Session-independent split"

paired_common_columns = [
    "evaluation_protocol",
    "metric",
    "metric_display_name",
    "seeds",
    "real_only_mean",
    "augmented_mean",
    "mean_paired_improvement",
    "improvement_standard_deviation",
    "confidence_interval_95_lower",
    "confidence_interval_95_upper",
    "augmented_wins",
    "ties",
    "augmented_losses"
]

final_paired_improvement_df = (
    pd.concat(
        [
            official_paired_df[
                paired_common_columns
            ],
            session_paired_df[
                paired_common_columns
            ]
        ],
        ignore_index=True
    )
)

metric_order = {
    metric_name: position
    for position, metric_name
    in enumerate(
        METRIC_DISPLAY_NAMES
    )
}

final_paired_improvement_df[
    "metric_order"
] = (
    final_paired_improvement_df[
        "metric"
    ].map(
        metric_order
    )
)

final_paired_improvement_df = (
    final_paired_improvement_df
    .sort_values([
        "metric_order",
        "evaluation_protocol"
    ])
    .drop(
        columns="metric_order"
    )
    .reset_index(drop=True)
)

final_paired_improvement_df.to_csv(
    FINAL_TABLE_DIR
    / "final_paired_augmentation_improvements.csv",
    index=False
)

display(
    final_paired_improvement_df.style.format({
        "real_only_mean":
            "{:.4f}",

        "augmented_mean":
            "{:.4f}",

        "mean_paired_improvement":
            "{:+.4f}",

        "improvement_standard_deviation":
            "{:.4f}",

        "confidence_interval_95_lower":
            "{:+.4f}",

        "confidence_interval_95_upper":
            "{:+.4f}"
    })
)

print(
    "The confidence intervals are "
    "descriptive because each protocol "
    "contains only five paired seeds."
)

In [ ]:
FINAL_PLOT_METRICS = [
    (
        "accuracy",
        "Accuracy"
    ),
    (
        "balanced_accuracy",
        "Balanced accuracy"
    ),
    (
        "macro_f1",
        "Macro-F1"
    ),
    (
        "bird_f1",
        "Bird F1"
    ),
    (
        "drone_recall",
        "Drone recall"
    ),
    (
        "roc_auc",
        "ROC-AUC"
    )
]

protocol_order = [
    "Official segment-level split",
    "Session-independent split"
]

configuration_order = [
    "real_only",
    "real_plus_synthetic"
]

configuration_labels = {
    "real_only":
        "10% real-only",

    "real_plus_synthetic":
        "10% real + synthetic 1:1"
}

configuration_colors = {
    "real_only":
        "#3569A8",

    "real_plus_synthetic":
        "#E67E22"
}

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9),
    constrained_layout=True
)

x_positions = np.arange(
    len(protocol_order)
)

bar_width = 0.34

for axis, (
    metric_name,
    metric_display_name
) in zip(
    axes.flat,
    FINAL_PLOT_METRICS
):
    for configuration_position, (
        configuration_name
    ) in enumerate(
        configuration_order
    ):
        plot_rows = (
            headline_results_df[
                headline_results_df[
                    "configuration"
                ] == configuration_name
            ]
            .set_index(
                "evaluation_protocol"
            )
            .reindex(
                protocol_order
            )
        )

        position_offset = (
            -bar_width / 2
            if configuration_position == 0
            else bar_width / 2
        )

        axis.bar(
            x_positions
            + position_offset,
            plot_rows[
                metric_name + "_mean"
            ],
            width=bar_width,
            yerr=plot_rows[
                metric_name + "_std"
            ],
            capsize=4,
            color=configuration_colors[
                configuration_name
            ],
            alpha=0.90,
            label=configuration_labels[
                configuration_name
            ]
        )

    axis.set_title(
        metric_display_name
    )

    axis.set_xticks(
        x_positions
    )

    axis.set_xticklabels([
        "Official\nsplit",
        "Session-\nindependent"
    ])

    axis.set_ylim(
        0.50,
        1.02
    )

    axis.set_ylabel(
        "Mean test score ± SD"
    )

    axis.grid(
        axis="y",
        alpha=0.25
    )

axes[0, 0].legend(
    loc="lower right",
    fontsize=8
)

fig.suptitle(
    "Multi-Seed Test Performance under "
    "Official and Session-Independent Evaluation",
    fontsize=15
)

headline_figure_path = (
    FINAL_FIGURE_DIR
    / "headline_multiseed_performance.png"
)

fig.savefig(
    headline_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Figure saved:",
    headline_figure_path.resolve()
)

## 7. Subtype and Target-Range Error Analysis

Subtype recall and target-range performance identify where augmentation helps and where limitations remain. Very small subtype supports are explicitly marked, and the results are treated as descriptive rather than population-level estimates.

In [ ]:
additional_artifact_paths = {
    "subtype_recall_comparison":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_augmentation"
            ],
            "subtype_recall_comparison.csv"
        ),

    "range_baseline_comparison":
        resolve_unique_artifact(
            EXPERIMENT_DIRECTORIES[
                "synthetic_augmentation"
            ],
            "range_baseline_comparison.csv"
        )
}

for (
    artifact_name,
    artifact_path
) in additional_artifact_paths.items():
    FINAL_ARTIFACT_PATHS[
        artifact_name
    ] = artifact_path

    final_result_tables[
        artifact_name
    ] = pd.read_csv(
        artifact_path
    )

    assert not (
        final_result_tables[
            artifact_name
        ].empty
    )

    assert not (
        final_result_tables[
            artifact_name
        ].isna().any().any()
    )

print(
    "Subtype comparison path:",
    additional_artifact_paths[
        "subtype_recall_comparison"
    ]
)

print(
    "Range comparison path:",
    additional_artifact_paths[
        "range_baseline_comparison"
    ]
)

print(
    "Final summary artifacts resolved:",
    len(
        FINAL_ARTIFACT_PATHS
    )
)

In [ ]:
subtype_comparison_df = (
    final_result_tables[
        "subtype_recall_comparison"
    ]
    .copy()
)

SUBTYPE_MODEL_COLUMNS = [
    "10% real-only",
    "10% real + synthetic 1:1",
    "25% real-only"
]

required_subtype_columns = {
    "true_target_group",
    "original_label",
    "samples",
    *SUBTYPE_MODEL_COLUMNS
}

missing_subtype_columns = (
    required_subtype_columns
    - set(
        subtype_comparison_df.columns
    )
)

if missing_subtype_columns:
    raise KeyError(
        "Missing subtype columns: "
        + str(
            sorted(
                missing_subtype_columns
            )
        )
    )

subtype_comparison_df[
    "support_note"
] = np.where(
    subtype_comparison_df[
        "samples"
    ] < 30,
    "Very limited support",
    "Adequate descriptive support"
)

subtype_comparison_df[
    "augmentation_change_vs_10_percent"
] = (
    subtype_comparison_df[
        "10% real + synthetic 1:1"
    ]
    - subtype_comparison_df[
        "10% real-only"
    ]
)

subtype_comparison_df[
    "augmentation_difference_from_25_percent"
] = (
    subtype_comparison_df[
        "10% real + synthetic 1:1"
    ]
    - subtype_comparison_df[
        "25% real-only"
    ]
)

subtype_comparison_df.to_csv(
    FINAL_TABLE_DIR
    / "final_subtype_recall_comparison.csv",
    index=False
)

display(
    subtype_comparison_df.style.format({
        "10% real-only":
            "{:.4f}",

        "10% real + synthetic 1:1":
            "{:.4f}",

        "25% real-only":
            "{:.4f}",

        "augmentation_change_vs_10_percent":
            "{:+.4f}",

        "augmentation_difference_from_25_percent":
            "{:+.4f}"
    })
)

subtype_plot_df = (
    subtype_comparison_df
    .sort_values([
        "true_target_group",
        "original_label"
    ])
    .reset_index(drop=True)
)

subtype_labels = [
    (
        f"{row.original_label} "
        f"(n={int(row.samples)})"
    )
    for row in subtype_plot_df.itertuples()
]

y_positions = np.arange(
    len(
        subtype_plot_df
    )
)

bar_height = 0.24

fig, axis = plt.subplots(
    figsize=(12, 8)
)

for model_position, (
    model_column
) in enumerate(
    SUBTYPE_MODEL_COLUMNS
):
    axis.barh(
        y_positions
        + (
            model_position - 1
        )
        * bar_height,
        subtype_plot_df[
            model_column
        ],
        height=bar_height,
        label=model_column
    )

axis.axvline(
    0.90,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="90% recall reference"
)

axis.set_yticks(
    y_positions
)

axis.set_yticklabels(
    subtype_labels
)

axis.set_xlim(
    0.0,
    1.03
)

axis.set_xlabel(
    "Official test recall"
)

axis.set_ylabel(
    "Original target subtype"
)

axis.set_title(
    "Subtype Recall: Real-Only versus "
    "Synthetic-Augmented Training"
)

axis.grid(
    axis="x",
    alpha=0.25
)

axis.legend(
    loc="lower right",
    fontsize=9
)

subtype_figure_path = (
    FINAL_FIGURE_DIR
    / "final_subtype_recall_comparison.png"
)

fig.savefig(
    subtype_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Figure saved:",
    subtype_figure_path.resolve()
)

In [ ]:
range_comparison_df = (
    final_result_tables[
        "range_baseline_comparison"
    ]
    .copy()
)

RANGE_ORDER = [
    "<30 m",
    "30–50 m",
    "50–75 m",
    ">75 m"
]

RANGE_MODEL_ORDER = [
    "10% real-only",
    "10% real + synthetic 1:1",
    "25% real-only"
]

range_comparison_df[
    "range_bin"
] = pd.Categorical(
    range_comparison_df[
        "range_bin"
    ],
    categories=RANGE_ORDER,
    ordered=True
)

range_comparison_df = (
    range_comparison_df
    .sort_values([
        "range_bin",
        "model"
    ])
    .reset_index(drop=True)
)

range_comparison_df.to_csv(
    FINAL_TABLE_DIR
    / "final_target_range_comparison.csv",
    index=False
)

display(
    range_comparison_df.style.format({
        "accuracy":
            "{:.4f}",

        "balanced_accuracy":
            "{:.4f}",

        "macro_f1":
            "{:.4f}",

        "bird_recall":
            "{:.4f}",

        "drone_recall":
            "{:.4f}",

        "roc_auc":
            "{:.4f}"
    })
)

RANGE_PLOT_METRICS = [
    (
        "balanced_accuracy",
        "Balanced accuracy"
    ),
    (
        "macro_f1",
        "Macro-F1"
    ),
    (
        "bird_recall",
        "Bird recall"
    )
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4.8),
    sharex=True,
    constrained_layout=True
)

for axis, (
    metric_name,
    metric_display_name
) in zip(
    axes,
    RANGE_PLOT_METRICS
):
    for model_name in RANGE_MODEL_ORDER:
        model_range_df = (
            range_comparison_df[
                range_comparison_df[
                    "model"
                ] == model_name
            ]
            .set_index(
                "range_bin"
            )
            .reindex(
                RANGE_ORDER
            )
        )

        axis.plot(
            RANGE_ORDER,
            model_range_df[
                metric_name
            ],
            marker="o",
            linewidth=2,
            label=model_name
        )

    axis.set_title(
        metric_display_name
    )

    axis.set_xlabel(
        "Target-range interval"
    )

    axis.set_ylim(
        0.40,
        1.02
    )

    axis.tick_params(
        axis="x",
        rotation=15
    )

    axis.grid(
        alpha=0.25
    )

axes[0].set_ylabel(
    "Official test score"
)

axes[0].legend(
    loc="lower left",
    fontsize=8
)

fig.suptitle(
    "Performance by Target Range",
    fontsize=14
)

range_figure_path = (
    FINAL_FIGURE_DIR
    / "final_target_range_comparison.png"
)

fig.savefig(
    range_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Figure saved:",
    range_figure_path.resolve()
)

## 8. Reproducibility and Completion Checks

The final key metrics, source-artifact checksums, generated tables, and figures are recorded in a reproducibility manifest. The completion check verifies that every required synthesis artifact exists and is non-empty.

In [ ]:
def get_headline_value(
    protocol,
    configuration,
    metric
):
    matching_rows = (
        headline_results_df[
            (
                headline_results_df[
                    "evaluation_protocol"
                ] == protocol
            )
            & (
                headline_results_df[
                    "configuration"
                ] == configuration
            )
        ]
    )

    assert len(
        matching_rows
    ) == 1

    return float(
        matching_rows.iloc[0][
            metric + "_mean"
        ]
    )


key_metric_records = []

for protocol_name in [
    "Official segment-level split",
    "Session-independent split"
]:
    for metric_name in [
        "balanced_accuracy",
        "macro_f1",
        "bird_f1"
    ]:
        real_only_value = (
            get_headline_value(
                protocol_name,
                "real_only",
                metric_name
            )
        )

        augmented_value = (
            get_headline_value(
                protocol_name,
                "real_plus_synthetic",
                metric_name
            )
        )

        key_metric_records.append({
            "experimental_context":
                protocol_name,

            "metric":
                METRIC_DISPLAY_NAMES[
                    metric_name
                ],

            "reference_value":
                real_only_value,

            "augmented_value":
                augmented_value,

            "absolute_change":
                augmented_value
                - real_only_value,

            "interpretation":
                "Five-seed mean test performance"
        })

far_range_rows = (
    range_comparison_df[
        range_comparison_df[
            "range_bin"
        ].astype(str) == ">75 m"
    ]
    .set_index(
        "model"
    )
)

for metric_name, metric_display_name in [
    (
        "balanced_accuracy",
        "Balanced accuracy"
    ),
    (
        "macro_f1",
        "Macro-F1"
    )
]:
    far_real_value = float(
        far_range_rows.loc[
            "10% real-only",
            metric_name
        ]
    )

    far_augmented_value = float(
        far_range_rows.loc[
            "10% real + synthetic 1:1",
            metric_name
        ]
    )

    key_metric_records.append({
        "experimental_context":
            "Official test targets beyond 75 m",

        "metric":
            metric_display_name,

        "reference_value":
            far_real_value,

        "augmented_value":
            far_augmented_value,

        "absolute_change":
            far_augmented_value
            - far_real_value,

        "interpretation":
            "Single-seed target-range analysis"
    })

final_key_metrics_df = pd.DataFrame(
    key_metric_records
)

final_key_metrics_df.to_csv(
    FINAL_TABLE_DIR
    / "final_key_metrics.csv",
    index=False
)

display(
    final_key_metrics_df.style.format({
        "reference_value":
            "{:.4f}",

        "augmented_value":
            "{:.4f}",

        "absolute_change":
            "{:+.4f}"
    })
)

In [ ]:
import hashlib
from datetime import datetime, timezone


def calculate_file_sha256(
    file_path
):
    digest = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:
        while True:
            block = file.read(
                1024 * 1024
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def project_relative_path(
    file_path
):
    resolved_path = (
        Path(
            file_path
        ).resolve()
    )

    try:
        return str(
            resolved_path.relative_to(
                PROJECT_ROOT.resolve()
            )
        )

    except ValueError:
        return str(
            resolved_path
        )


source_artifact_manifest = {}

for (
    artifact_name,
    artifact_path
) in FINAL_ARTIFACT_PATHS.items():
    source_artifact_manifest[
        artifact_name
    ] = {
        "relative_path":
            project_relative_path(
                artifact_path
            ),

        "size_bytes":
            int(
                artifact_path
                .stat()
                .st_size
            ),

        "sha256":
            calculate_file_sha256(
                artifact_path
            )
    }

final_synthesis_manifest = {
    "experiment":
        "final_results_synthesis",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "analysis_mode":
        "saved_artifacts_only",

    "safety": {
        "model_training_performed":
            False,

        "new_test_inference_performed":
            False,

        "new_threshold_selection_performed":
            False
    },

    "model_seeds": [
        42,
        52,
        62,
        72,
        82
    ],

    "selected_synthetic_to_real_ratio":
        selected_ratio,

    "ratio_selection_partition":
        "validation",

    "source_artifact_count":
        len(
            FINAL_ARTIFACT_PATHS
        ),

    "core_results": {
        "official_split": {
            "real_only_macro_f1_mean":
                get_headline_value(
                    "Official segment-level split",
                    "real_only",
                    "macro_f1"
                ),

            "augmented_macro_f1_mean":
                get_headline_value(
                    "Official segment-level split",
                    "real_plus_synthetic",
                    "macro_f1"
                ),

            "real_only_bird_f1_mean":
                get_headline_value(
                    "Official segment-level split",
                    "real_only",
                    "bird_f1"
                ),

            "augmented_bird_f1_mean":
                get_headline_value(
                    "Official segment-level split",
                    "real_plus_synthetic",
                    "bird_f1"
                )
        },

        "session_independent_split": {
            "real_only_macro_f1_mean":
                get_headline_value(
                    "Session-independent split",
                    "real_only",
                    "macro_f1"
                ),

            "augmented_macro_f1_mean":
                get_headline_value(
                    "Session-independent split",
                    "real_plus_synthetic",
                    "macro_f1"
                ),

            "real_only_bird_f1_mean":
                get_headline_value(
                    "Session-independent split",
                    "real_only",
                    "bird_f1"
                ),

            "augmented_bird_f1_mean":
                get_headline_value(
                    "Session-independent split",
                    "real_plus_synthetic",
                    "bird_f1"
                )
        }
    },

    "source_artifacts":
        source_artifact_manifest
}

FINAL_MANIFEST_PATH = (
    FINAL_SYNTHESIS_DIR
    / "final_synthesis_manifest.json"
)

with open(
    FINAL_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_synthesis_manifest,
        file,
        indent=2
    )

print(
    "Final synthesis manifest saved:",
    FINAL_MANIFEST_PATH.resolve()
)

print(
    "Source artifacts recorded:",
    len(
        source_artifact_manifest
    )
)

In [ ]:
required_final_tables = [
    "artifact_directory_inventory.csv",
    "headline_multiseed_results.csv",
    "augmentation_gain_preservation.csv",
    "final_real_data_learning_curve.csv",
    "synthetic_augmentation_data_efficiency.csv",
    "final_synthetic_quality_summary.csv",
    "final_synthetic_alignment_summary.csv",
    "real_vs_synthetic_distribution.csv",
    "final_ratio_ablation_summary.csv",
    "final_paired_augmentation_improvements.csv",
    "final_subtype_recall_comparison.csv",
    "final_target_range_comparison.csv",
    "final_key_metrics.csv"
]

required_final_figures = [
    "learning_curve_and_augmentation.png",
    "synthetic_ratio_ablation.png",
    "headline_multiseed_performance.png",
    "final_subtype_recall_comparison.png",
    "final_target_range_comparison.png"
]

completion_records = []

for file_name in required_final_tables:
    file_path = (
        FINAL_TABLE_DIR
        / file_name
    )

    completion_records.append({
        "artifact_type":
            "table",

        "file_name":
            file_name,

        "exists":
            file_path.exists(),

        "size_bytes":
            (
                file_path.stat().st_size
                if file_path.exists()
                else 0
            )
    })

for file_name in required_final_figures:
    file_path = (
        FINAL_FIGURE_DIR
        / file_name
    )

    completion_records.append({
        "artifact_type":
            "figure",

        "file_name":
            file_name,

        "exists":
            file_path.exists(),

        "size_bytes":
            (
                file_path.stat().st_size
                if file_path.exists()
                else 0
            )
    })

completion_records.append({
    "artifact_type":
        "manifest",

    "file_name":
        FINAL_MANIFEST_PATH.name,

    "exists":
        FINAL_MANIFEST_PATH.exists(),

    "size_bytes":
        (
            FINAL_MANIFEST_PATH
            .stat()
            .st_size
            if FINAL_MANIFEST_PATH.exists()
            else 0
        )
})

final_completion_df = pd.DataFrame(
    completion_records
)

display(
    final_completion_df
)

assert (
    final_completion_df[
        "exists"
    ].all()
)

assert (
    final_completion_df[
        "size_bytes"
    ] > 0
).all()

print(
    "FINAL SYNTHESIS COMPLETE"
)

print(
    "Tables:",
    len(
        required_final_tables
    )
)

print(
    "Figures:",
    len(
        required_final_figures
    )
)

print(
    "Manifest:",
    FINAL_MANIFEST_PATH.name
)

print(
    "No model training or new test "
    "inference was performed."
)

## 9. Final Conclusions

This study investigated whether controlled transformation-based synthetic augmentation can improve low-data bird–drone classification using 77 GHz FMCW radar micro-Doppler representations.

### Main findings

1. **Real-data quantity strongly affects minority-class performance.** Increasing the balanced real training set from 10% to 25% raised macro-F1 from 0.8082 to 0.9579 and bird F1 from 0.6797 to 0.9277. Improvements became smaller between 25%, 50%, and 100%, indicating diminishing returns.

2. **Synthetic augmentation substantially improved the 10% real-data model.** In the reference-seed experiment, 1:1 augmentation increased macro-F1 from 0.8082 to 0.9378 and bird F1 from 0.6797 to 0.8934. It recovered 86.58% of the 10%-to-25% real-data macro-F1 gap and 86.16% of the bird-F1 gap.

3. **The benefit was robust across model seeds.** Under the official split, mean macro-F1 increased from 0.8003 to 0.9209 and mean bird F1 increased from 0.6749 to 0.8654 across five paired seeds. Augmentation also reduced performance variability and improved worst-case results.

4. **The selected synthetic-to-real ratio was 1:1.** Ratio selection used validation results only. The 0.5:1 ratio already produced a substantial improvement, but the 1:1 ratio achieved the highest mean validation macro-F1, balanced accuracy, and accuracy.

5. **The augmentation benefit survived strict session separation.** On unseen recording sessions, mean macro-F1 increased from 0.8485 to 0.9615 and bird F1 increased from 0.7427 to 0.9314. The bird-F1 augmentation gain was +0.1887, compared with +0.1905 under the official split.

6. **Synthetic augmentation improved difficult operating conditions.** For targets beyond 75 m, macro-F1 increased from 0.4585 to 0.7757 and balanced accuracy increased from 0.7491 to 0.8677.

### Important limitations

- Synthetic observations are transformed children of real training samples and are not independent physical radar acquisitions.
- Only five paired model seeds were evaluated, producing wide confidence intervals for some improvements.
- The session-independent experiment used one fixed session assignment.
- Pigeon and raven contain very limited test support, so their subtype results must not be generalized.
- Heron recall decreased after augmentation, demonstrating that aggregate improvement does not guarantee improvement for every subtype.
- Long-range performance improved but remained below the 25% real-only model.
- Official and session-independent protocols use different test observations; comparisons between their absolute scores are descriptive.

### Overall conclusion

Controlled signal-domain synthetic augmentation is an effective data-efficiency strategy for this radar classification problem. It substantially improves minority-class recognition, reduces sensitivity to model initialization, and retains its benefit when evaluated on unseen recording sessions. It should be considered a complement to additional real radar acquisition rather than a replacement for genuinely independent measurements.

## 10. LaTeX Report Table Export

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Assumes the notebook is executed from the notebooks directory.
PROJECT_ROOT = Path("..").resolve()

FINAL_TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "final_results_synthesis"
    / "tables"
)

REPORT_TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "report_assets"
    / "tables"
)

REPORT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Input table directory:", FINAL_TABLE_DIR)
print("Report table directory:", REPORT_TABLE_DIR)


def save_latex_table(
    dataframe,
    file_name,
    caption,
    label,
    resize=False
):
    """
    Save a complete LaTeX table environment.
    Strings may contain LaTeX commands, so escape=False is intentional.
    """
    dataframe = dataframe.copy()

    column_format = (
        "l"
        + "c" * (len(dataframe.columns) - 1)
    )

    tabular = dataframe.to_latex(
        index=False,
        escape=False,
        column_format=column_format
    )

    if resize:
        table_body = (
            "\\resizebox{\\textwidth}{!}{%\n"
            + tabular
            + "}\n"
        )
    else:
        table_body = tabular

    complete_table = (
        "\\begin{table}[htbp]\n"
        "\\centering\n"
        "\\small\n"
        f"{table_body}"
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        "\\end{table}\n"
    )

    output_path = (
        REPORT_TABLE_DIR
        / file_name
    )

    output_path.write_text(
        complete_table,
        encoding="utf-8"
    )

    print("Saved:", output_path.name)

    return output_path

In [ ]:
dataset_specification_df = pd.DataFrame({
    "Property": [
        "Radar system",
        "Carrier frequency",
        "Pulse repetition frequency",
        "Measurement sessions",
        "Raw radar segments",
        "Raw segment representation",
        "Retained feature-axis interval",
        "Processed CNN input",
        "Input normalization",
        "Binary label encoding"
    ],
    "Value": [
        "SAAB SIRS 1600 FMCW radar",
        "77 GHz",
        "17 kHz",
        "130",
        "75,868",
        "$5 \\times 256$ complex samples",
        "Bins 54--203, giving 150 retained bins",
        "$5 \\times 150 \\times 1$",
        "Per-segment values scaled to $[0,1]$",
        "Bird $=0$; drone $=1$"
    ]
})

save_latex_table(
    dataset_specification_df,
    "table_01_dataset_specifications.tex",
    (
        "Radar dataset and processed-input "
        "specifications."
    ),
    "tab:dataset-specifications"
)

In [ ]:
dataset_composition_df = pd.DataFrame({
    "Target group": [
        "Bird",
        "Drone",
        "Excluded: human and corner reflector",
        "Complete dataset"
    ],
    "Sessions": [
        56,
        44,
        30,
        130
    ],
    "Total segments": [
        7792,
        58768,
        9308,
        75868
    ],
    "Edge segments": [
        56,
        11,
        0,
        67
    ],
    "Usable segments": [
        7736,
        58757,
        9308,
        75801
    ]
})

for column in [
    "Sessions",
    "Total segments",
    "Edge segments",
    "Usable segments"
]:
    dataset_composition_df[column] = (
        dataset_composition_df[column]
        .map(lambda value: f"{value:,}")
    )

save_latex_table(
    dataset_composition_df,
    "table_02_dataset_composition.tex",
    (
        "Composition of the complete dataset "
        "before binary bird--drone classification."
    ),
    "tab:dataset-composition"
)

In [ ]:
evaluation_partition_df = pd.DataFrame({
    "Protocol": [
        "Official segment-level",
        "Official segment-level",
        "Official segment-level",
        "Session-independent",
        "Session-independent",
        "Session-independent"
    ],
    "Partition": [
        "Training",
        "Validation",
        "Test",
        "Training",
        "Validation",
        "Test"
    ],
    "Samples": [
        52509,
        6988,
        6996,
        53011,
        6943,
        6539
    ],
    "Birds": [
        5749,
        990,
        997,
        6236,
        812,
        688
    ],
    "Drones": [
        46760,
        5998,
        5999,
        46775,
        6131,
        5851
    ],
    "Sessions": [
        "Not isolated",
        "Not isolated",
        "Not isolated",
        "80",
        "10",
        "10"
    ]
})

for column in [
    "Samples",
    "Birds",
    "Drones"
]:
    evaluation_partition_df[column] = (
        evaluation_partition_df[column]
        .map(lambda value: f"{value:,}")
    )

save_latex_table(
    evaluation_partition_df,
    "table_03_evaluation_partitions.tex",
    (
        "Composition of the official and "
        "session-independent evaluation partitions."
    ),
    "tab:evaluation-partitions",
    resize=True
)

In [ ]:
synthetic_transformation_df = pd.DataFrame({
    "Transformation": [
        "Feature-axis translation",
        "Amplitude scaling",
        "Contrast adjustment",
        "Additive Gaussian noise",
        "Partial feature masking",
        "Final clipping"
    ],
    "Configuration": [
        "$-5$ to $+5$ bins",
        "$0.90$ to $1.10$",
        "$0.90$ to $1.10$",
        "Standard deviation $0.002$ to $0.015$",
        (
            "Probability $0.30$; width 2--6 bins; "
            "sample-mean fill"
        ),
        "$[0,1]$"
    ],
    "Boundary or reference rule": [
        "Nearest boundary value; no circular wrapping",
        "Applied multiplicatively",
        "Applied around the sample mean",
        "Zero-mean noise",
        "Contiguous mask along the feature axis",
        "Preserves the preprocessing interval"
    ]
})

save_latex_table(
    synthetic_transformation_df,
    "table_04_synthetic_transformations.tex",
    (
        "Controlled transformations used to generate "
        "synthetic training samples."
    ),
    "tab:synthetic-transformations",
    resize=True
)

In [ ]:
model_configuration_df = pd.DataFrame({
    "Component": [
        "Input",
        "Convolution block 1",
        "Convolution block 2",
        "Convolution block 3",
        "Feature aggregation",
        "Dense layer",
        "Output",
        "Trainable architecture size",
        "Optimizer",
        "Loss",
        "Batch size",
        "Maximum epochs",
        "Early stopping",
        "Learning-rate reduction",
        "Decision threshold"
    ],
    "Configuration": [
        "$5 \\times 150 \\times 1$",
        (
            "16 filters, $3 \\times 7$ kernel, "
            "batch normalization, ReLU, $1 \\times 2$ pooling"
        ),
        (
            "32 filters, $3 \\times 5$ kernel, "
            "batch normalization, ReLU, $1 \\times 2$ pooling"
        ),
        (
            "64 filters, $3 \\times 3$ kernel, "
            "batch normalization, ReLU, $1 \\times 2$ pooling"
        ),
        "Global average pooling",
        (
            "32 ReLU units, $L_2=10^{-4}$, "
            "dropout $=0.30$"
        ),
        "One sigmoid unit",
        "29,121 parameters",
        "Adam, initial learning rate $10^{-3}$",
        "Binary cross-entropy",
        "64",
        "50",
        "Validation loss, patience 8",
        (
            "Factor 0.5, patience 4, "
            "minimum learning rate $10^{-6}$"
        ),
        (
            "Maximum validation macro-F1, with "
            "balanced accuracy as tie-breaker"
        )
    ]
})

save_latex_table(
    model_configuration_df,
    "table_05_model_configuration.tex",
    (
        "Compact convolutional neural-network "
        "architecture and training configuration."
    ),
    "tab:model-configuration",
    resize=True
)

In [ ]:
experimental_design_df = pd.DataFrame({
    "Experiment": [
        "Real-data learning curve",
        "Initial augmentation comparison",
        "Multi-seed robustness",
        "Synthetic-ratio ablation",
        "Session-independent evaluation"
    ],
    "Training conditions": [
        "10\\%, 25\\%, 50\\%, and 100\\% balanced real data",
        "10\\% real-only versus 10\\% real plus synthetic 1:1",
        (
            "Real-only versus augmented using seeds "
            "42, 52, 62, 72, and 82"
        ),
        (
            "Synthetic-to-real ratios 0:1, 0.5:1, and 1:1 "
            "across five seeds"
        ),
        (
            "10\\% session-independent real-only versus "
            "real plus synthetic 1:1 across five seeds"
        )
    ],
    "Selection or evaluation rule": [
        "Same official validation and test partitions",
        "Threshold selected on validation data",
        "Paired seeds and locked validation thresholds",
        "Ratio selected using validation results only",
        (
            "Disjoint training, validation, and test "
            "recording sessions"
        )
    ]
})

save_latex_table(
    experimental_design_df,
    "table_06_experimental_design.tex",
    "Summary of the principal experiments.",
    "tab:experimental-design",
    resize=True
)

In [ ]:
quality_path = (
    FINAL_TABLE_DIR
    / "final_synthetic_quality_summary.csv"
)

synthetic_quality_raw_df = pd.read_csv(
    quality_path
)

synthetic_quality_df = (
    synthetic_quality_raw_df[
        [
            "quality_indicator",
            "value"
        ]
    ]
    .rename(columns={
        "quality_indicator": "Quality indicator",
        "value": "Value"
    })
)

def format_quality_value(row):
    indicator = row["Quality indicator"]
    value = float(row["Value"])

    if (
        indicator
        == "Samples below 0.80 aligned correlation"
    ):
        return str(int(round(value)))

    return f"{value:.4f}"


synthetic_quality_df["Value"] = (
    synthetic_quality_df.apply(
        format_quality_value,
        axis=1
    )
)

save_latex_table(
    synthetic_quality_df,
    "table_07_synthetic_quality.tex",
    (
        "Quality indicators for the generated "
        "synthetic radar samples."
    ),
    "tab:synthetic-quality"
)

In [ ]:
headline_path = (
    FINAL_TABLE_DIR
    / "headline_multiseed_results.csv"
)

headline_raw_df = pd.read_csv(
    headline_path
)


def mean_std_text(row, metric):
    return (
        f"${row[f'{metric}_mean']:.4f}"
        f" \\pm "
        f"{row[f'{metric}_std']:.4f}$"
    )


headline_table_records = []

for _, row in headline_raw_df.iterrows():
    protocol = str(
        row["evaluation_protocol"]
    )

    protocol = protocol.replace(
        "Official segment-level split",
        "Official"
    ).replace(
        "Session-independent split",
        "Session-independent"
    )

    training = str(
        row["configuration_display_name"]
    )

    training = training.replace(
        "10% session-independent real + synthetic 1:1",
        "Real + synthetic"
    ).replace(
        "10% session-independent real-only",
        "Real-only"
    ).replace(
        "10% real + synthetic 1:1",
        "Real + synthetic"
    ).replace(
        "10% real-only",
        "Real-only"
    )

    headline_table_records.append({
        "Protocol": protocol,
        "Training": training,
        "Accuracy": mean_std_text(
            row,
            "accuracy"
        ),
        "Balanced accuracy": mean_std_text(
            row,
            "balanced_accuracy"
        ),
        "Macro-F1": mean_std_text(
            row,
            "macro_f1"
        ),
        "Bird F1": mean_std_text(
            row,
            "bird_f1"
        ),
        "Drone recall": mean_std_text(
            row,
            "drone_recall"
        ),
        "ROC-AUC": mean_std_text(
            row,
            "roc_auc"
        )
    })

headline_table_df = pd.DataFrame(
    headline_table_records
)

save_latex_table(
    headline_table_df,
    "table_08_headline_results.tex",
    (
        "Mean test performance across five model seeds. "
        "Values are reported as mean $\\pm$ standard deviation."
    ),
    "tab:headline-results",
    resize=True
)

print(
    "\nAll principal report tables were generated."
)